In [1]:
import os
import json
import cv2
import numpy as np

DATA_PATH = "Data/Hurricane"
IMG_SIZE = 224

X_cnn = []
y_cnn = []

for json_file in os.listdir(f"{DATA_PATH}/JSON"):
    
    with open(f"{DATA_PATH}/JSON/{json_file}") as f:
        data = json.load(f)

    frame_path = os.path.join(DATA_PATH, data["Frame_Name"])
    image = cv2.imread(frame_path)

    if image is None:
        print("Could not read image:", frame_path)
        continue

    # OpenCV loads as BGR, convert to RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0

    for building in data["Buildings"]:
        
        mask_filename = building[2]
        label = building[3]

        mask_path = os.path.join(DATA_PATH, "MASK", mask_filename)

        if not os.path.exists(mask_path):
            continue

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if mask is None:
            print("Could not read mask:", mask_path)
            continue

        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
        mask = mask / 255.0
        mask = np.expand_dims(mask, axis=-1)

        # Combine RGB image + mask channel
        combined = np.concatenate([image, mask], axis=-1)

        X_cnn.append(combined)
        y_cnn.append(label)

X_cnn = np.array(X_cnn, dtype=np.float32)
y_cnn = np.array(y_cnn, dtype=np.int64)

print("X_cnn shape:", X_cnn.shape)
print("y_cnn shape:", y_cnn.shape)
print("Labels:", sorted(set(y_cnn)))

X_cnn shape: (1458, 224, 224, 4)
y_cnn shape: (1458,)
Labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [2]:
from sklearn.model_selection import train_test_split

X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(
    X_cnn,
    y_cnn,
    test_size=0.2,
    random_state=42,
    stratify=y_cnn
)

print("X_train_cnn:", X_train_cnn.shape)
print("X_test_cnn:", X_test_cnn.shape)
print("y_train_cnn:", y_train_cnn.shape)
print("y_test_cnn:", y_test_cnn.shape)

X_train_cnn: (1166, 224, 224, 4)
X_test_cnn: (292, 224, 224, 4)
y_train_cnn: (1166,)
y_test_cnn: (292,)


In [4]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train_cnn)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_cnn
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, class_weights_array)
}

print(class_weights)

{0: 1.4502487562189055, 1: 0.9210110584518167, 2: 1.237791932059448, 3: 0.5238095238095238, 4: 0.9433656957928802, 5: 2.2337164750957856}


In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models

num_classes = len(np.unique(y_cnn))

cnn = models.Sequential([
    layers.Input(shape=(224, 224, 4)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(num_classes, activation="softmax")
])

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │         1,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     4,718,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,993,638 (19.05 MB)

 Trainable params: 4,993,638 (19.05 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = cnn.fit(
    X_train_cnn,
    y_train_cnn,
    validation_split=0.2,
    epochs=30,
    batch_size=16,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 10s 152ms/step - accuracy: 0.2264 - loss: 1.7934 - val_accuracy: 0.2863 - val_loss: 1.7749
Epoch 2/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 9s 153ms/step - accuracy: 0.2650 - loss: 1.7506 - val_accuracy: 0.1923 - val_loss: 1.7634
Epoch 3/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 9s 157ms/step - accuracy: 0.3348 - loss: 1.6291 - val_accuracy: 0.2521 - val_loss: 1.7208
Epoch 4/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 9s 154ms/step - accuracy: 0.4045 - loss: 1.5178 - val_accuracy: 0.3205 - val_loss: 1.6368
Epoch 5/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 9s 157ms/step - accuracy: 0.4635 - loss: 1.3749 - val_accuracy: 0.3590 - val_loss: 1.6108
Epoch 6/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 10s 167ms/step - accuracy: 0.5161 - loss: 1.2435 - val_accuracy: 0.3462 - val_loss: 1.6671
Epoch 7/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 9s 160ms/step - accuracy: 0.5418 - loss: 1.1021 - val_accuracy: 0.4145 - val_loss: 1.7303
Epoch 8/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 10s 162ms/step - accuracy: 0.6041 - loss: 0.9677 - val_accuracy: